## Machine Learning for Linear Regression, Random Forest & Gradient Boosting

### Load Original Dataset

In [7]:
import pandas as pd

raw_df = pd.read_csv("stock_datasets/RELIANCE.csv")

### Create Working Copy

In [8]:
df = raw_df.copy()

### Inspect Dataset

In [9]:
print(df.head())

print(df.info())

print(df.shape)
print(df.dtypes)

         Date        Open        High         Low       Close    Volume
0  2015-01-01  189.657416  190.877140  189.090335  189.999786   2963643
1  2015-01-02  190.042601  191.743813  189.229456  189.496933   7331366
2  2015-01-05  189.379224  190.641759  187.046752  187.421234  10103941
3  2015-01-06  186.169433  186.811392  178.037884  178.915237  18627980
4  2015-01-07  179.129214  183.772746  179.107815  182.809799  20720312
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2786 entries, 0 to 2785
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    2786 non-null   object 
 1   Open    2786 non-null   float64
 2   High    2786 non-null   float64
 3   Low     2786 non-null   float64
 4   Close   2786 non-null   float64
 5   Volume  2786 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 130.7+ KB
None
(2786, 6)
Date       object
Open      float64
High      float64
Low       float64
Close     float

### Check Missing Values

In [10]:
print(df.isnull().sum())

Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64


### Remove Missing Values

In [11]:
df.dropna(inplace=True)

### Check Duplicate Rows

In [12]:
print(df.duplicated().sum())

0


### Remove Duplicate Rows

In [13]:
df.drop_duplicates(inplace=True)

### Convert Date Column : Convert Date from object/string to datetime.

In [14]:
df["Date"] = pd.to_datetime(df["Date"])

### Sort Data Chronologically : important for stock prediction

In [15]:
df = df.sort_values(by="Date")

df = df.reset_index(drop=True)

### Create Date Features : Extract useful numerical features from Date

In [16]:
df["Year"] = df["Date"].dt.year

df["Month"] = df["Date"].dt.month

df["Day"] = df["Date"].dt.day

df["DayOfWeek"] = df["Date"].dt.dayofweek

### Drop Original Date Column : After extracting features, remove raw Date column

In [17]:
df.drop(columns=["Date"], inplace=True)

### Check New Dataset Column

In [18]:
df.head()

,Open,High,Low,Close,Volume,Year,Month,Day,DayOfWeek
0,189.657416,190.877140,189.090335,189.999786,2963643,2015,1,1,3
1,190.042601,191.743813,189.229456,189.496933,7331366,2015,1,2,4
2,189.379224,190.641759,187.046752,187.421234,10103941,2015,1,5,0
3,186.169433,186.811392,178.037884,178.915237,18627980,2015,1,6,1
4,179.129214,183.772746,179.107815,182.809799,20720312,2015,1,7,2


### Keep Required Columns Only

In [19]:
df = df[[
    "Open", "High", "Low", "Close", "Volume",
    "Year", "Month", "Day", "DayOfWeek"
]]

### Create Return Feature

In [20]:
df["Return"] = df["Close"].pct_change()

### Create Moving Averages

In [21]:
df["MA10"] = df["Close"].rolling(10).mean()

df["MA50"] = df["Close"].rolling(50).mean()

### Create Volatility Feature

In [22]:
df["Volatility"] = df["Return"].rolling(10).std()

### Create Momentum Feature

In [23]:
df["Momentum"] = df["Close"] - df["Close"].shift(10)

### Create Lag Features : important for stock prediction

In [24]:
df["Lag1"] = df["Close"].shift(1)

df["Lag2"] = df["Close"].shift(2)

df["Lag3"] = df["Close"].shift(3)

### Create Price Range Feature

In [25]:
df["Range"] = df["High"] - df["Low"]

### Create EMA Feature

In [26]:
df["EMA10"] = df["Close"].ewm(span=10).mean()

### Create RSI Feature

In [27]:
delta = df["Close"].diff()

gain = delta.where(delta > 0, 0)

loss = -delta.where(delta < 0, 0)

avg_gain = gain.rolling(14).mean()

avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df["RSI"] = 100 - (100 / (1 + rs))

### Create Target Variable : Predict next-day closing price.

In [28]:
df["Target"] = df["Close"].shift(-1)

### Remove Generated NaN Values : Rolling and shifting create NaN rows

In [29]:
df.dropna(inplace=True)

### Final Dataset Validation

In [30]:
print(df.info())

print(df.isnull().sum())

print(df.head())

<class 'pandas.core.frame.DataFrame'>
Index: 2736 entries, 49 to 2784
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Open        2736 non-null   float64
 1   High        2736 non-null   float64
 2   Low         2736 non-null   float64
 3   Close       2736 non-null   float64
 4   Volume      2736 non-null   int64  
 5   Year        2736 non-null   int32  
 6   Month       2736 non-null   int32  
 7   Day         2736 non-null   int32  
 8   DayOfWeek   2736 non-null   int32  
 9   Return      2736 non-null   float64
 10  MA10        2736 non-null   float64
 11  MA50        2736 non-null   float64
 12  Volatility  2736 non-null   float64
 13  Momentum    2736 non-null   float64
 14  Lag1        2736 non-null   float64
 15  Lag2        2736 non-null   float64
 16  Lag3        2736 non-null   float64
 17  Range       2736 non-null   float64
 18  EMA10       2736 non-null   float64
 19  RSI         2736 non-null   flo

### Save Processed Dataset

In [31]:
df.to_csv("stock_datasets/RELIANCE_processed.csv", index=False)

## Deep Learning for LSTM

### Create New Copy

In [32]:
lstm_df = raw_df.copy()

### Inspect Dataset

In [33]:
print(lstm_df.head())

print(lstm_df.info())

         Date        Open        High         Low       Close    Volume
0  2015-01-01  189.657416  190.877140  189.090335  189.999786   2963643
1  2015-01-02  190.042601  191.743813  189.229456  189.496933   7331366
2  2015-01-05  189.379224  190.641759  187.046752  187.421234  10103941
3  2015-01-06  186.169433  186.811392  178.037884  178.915237  18627980
4  2015-01-07  179.129214  183.772746  179.107815  182.809799  20720312
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2786 entries, 0 to 2785
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    2786 non-null   object 
 1   Open    2786 non-null   float64
 2   High    2786 non-null   float64
 3   Low     2786 non-null   float64
 4   Close   2786 non-null   float64
 5   Volume  2786 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 130.7+ KB
None


### Handle Missing Values

In [34]:
print(lstm_df.isnull().sum())

lstm_df.dropna(inplace=True)

Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64


### Remove Duplicate Rows

In [35]:
print(lstm_df.duplicated().sum())

lstm_df.drop_duplicates(inplace=True)

0


### Convert Date Column

In [36]:
lstm_df["Date"] = pd.to_datetime(lstm_df["Date"])

### Sort Chronologically : IMPORTANT for LSTM

In [37]:
lstm_df = lstm_df.sort_values(by="Date")

lstm_df = lstm_df.reset_index(drop=True)

### Keep Required Columns

In [38]:
lstm_df = lstm_df[["Date", "Close"]]

### Save Unscaled Backup

In [39]:
lstm_df.to_csv(
    "stock_datasets/RELIANCE_lstm_unscaled.csv",
    index=False
)

### Scale Data : LSTM requires scaling without scaling training unstable

In [40]:
### use MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))

scaled_data = scaler.fit_transform(lstm_df[["Close"]])

### Create Sequence Windows : MOST IMPORTANT LSTM preprocessing step, LSTM learns previous timesteps

In [41]:
import numpy as np

X = []
y = []

window_size = 60

for i in range(window_size, len(scaled_data)):
    
    X.append(scaled_data[i-window_size:i])
    
    y.append(scaled_data[i])

X = np.array(X)
y = np.array(y)

### Check Shapes

In [42]:
print(X.shape)

print(y.shape)

(2726, 60, 1)
(2726, 1)


### Train-Test Split : For time-series: NO SHUFFLING

In [43]:
split = int(len(X) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

### Save Processed Arrays

In [44]:
np.save("stock_datasets/X_train.npy", X_train)

np.save("stock_datasets/X_test.npy", X_test)

np.save("stock_datasets/y_train.npy", y_train)

np.save("stock_datasets/y_test.npy", y_test)